# 🚀 Automated Parallel PSA Generator Pipeline

This notebook runs the complete 4-way parallel PSA generator pipeline (English -> Swahili, Somali, Luo) on Google Colab's GPU runtime.

### Step 0: Clone Repository
Clone the repository to fetch the required generator scripts and templates, then change into the project directory. If the directory already exists, it will pull the latest updates.

In [ ]:
import os
if not os.path.exists('public-service-anouncement-MT'):
    !git clone https://github.com/SamAbr/public-service-anouncement-MT.git
    %cd public-service-anouncement-MT
else:
    print('Repository directory already exists. Pulling latest updates...')
    %cd public-service-anouncement-MT
    !git pull

### Step 1: Install Dependencies
Install the required translation, tracking, and language ID libraries.

In [ ]:
!pip install transformers sentencepiece tqdm pandas torch fasttext nltk

### Step 2: Generate English PSAs & Download for Review
Generate 50,000 unique, validated, and tag-cohered English announcements. A sample will be displayed below, and the complete CSV file will be downloaded to your computer automatically for your review.

In [ ]:
!python generate_english_only.py --size 50000 --output output/english_psas.csv

# Show a sample of the generated English PSAs for quick review
import pandas as pd
from google.colab import files

df_sample = pd.read_csv('output/english_psas.csv')
print("\n--- Sample of Generated English PSAs ---")
display(df_sample.head(10))

# Download the file automatically for full local review
print("\nDownloading 'english_psas.csv' to your computer for review...")
files.download('output/english_psas.csv')

### Step 3: Run Sequential Translation Pipeline
Translate the English seed set to Swahili, Somali, and Luo sequentially on the GPU with quality filters (LangID and ChrF round-trip).

**Note:** Proceed to run this step *after* you have reviewed and approved the English dataset downloaded in Step 2.

In [ ]:
!python translate_colab.py --input output/english_psas.csv --output output/psa_parallel_dataset.csv --batch-size 128

### Step 4: Download Parallel Dataset
Download the final parallel CSV file to your local computer.

In [ ]:
from google.colab import files
files.download('output/psa_parallel_dataset.csv')

### Step 5: Save, Commit, and Push to GitHub
Upload the generated CSV file directly to your GitHub repository. 

**Note:** You will need to enter your GitHub Personal Access Token (PAT) securely when prompted.

In [ ]:
import getpass

# 1. Prompt for authentication details securely
git_name = input("Enter your Git username: ")
git_email = input("Enter your Git email: ")
git_token = getpass.getpass("Enter your GitHub Personal Access Token (PAT): ")

# 2. Configure Git in the VM
!git config --global user.name "{git_name}"
!git config --global user.email "{git_email}"

# 3. Configure the push URL with the access token
!git remote set-url origin https://{git_token}@github.com/SamAbr/public-service-anouncement-MT.git

# 4. Stage, commit, and push the CSV file
!git add output/psa_parallel_dataset.csv output/english_psas.csv
!git commit -m "Upload generated parallel datasets from Colab GPU run"
!git push origin main